# Experimentieren mit der Architektur des NN

Dieses Notebook vergleicht verschiedene neuronale Netzwerkarchitekturen auf denselben vorbereiteten Daten, ohne die zugrunde liegende Datenlogik zu verändern.

# 1. Setup & Imports
In dieser Sektion werden alle benötigten Bibliotheken geladen und Warnungen reduziert, damit das Notebook sauber und konsistent läuft.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import sklearn
import tensorflow.keras as keras
from sklearn.datasets import fetch_california_housing

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

2026-03-20 14:29:33.973978: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-20 14:29:33.980980: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 14:29:34.272002: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 14:29:36.848720: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

# 2. Datenvorbereitung
Die California-Housing-Daten werden bereinigt, transformiert und in Trainings- und Testdaten aufgeteilt.

In [ ]:
SEED = 42
SF_COORDS = (37.7749, -122.4194)
LA_COORDS = (34.0522, -118.2437)
SJ_COORDS = (37.3362, -121.8833)


def prepare_data():
    data = fetch_california_housing()
    df = pd.DataFrame(data=data.data, columns=data.feature_names)
    df[data.target_names[0]] = data.target

    mask_cutoff = (
        (df["MedHouseVal"] < df["MedHouseVal"].max())
        & (df["HouseAge"] < df["HouseAge"].max())
        & (df["MedInc"] < df["MedInc"].max())
    )
    df_clean = df[mask_cutoff].copy()

    clip_columns = ["AveRooms", "AveBedrms", "AveOccup", "Population"]
    for column in clip_columns:
        lower_bound = df_clean[column].quantile(0.01)
        upper_bound = df_clean[column].quantile(0.99)
        df_clean[column] = df_clean[column].clip(lower=lower_bound, upper=upper_bound)

    df_clean["BedrmsPerRoom"] = df_clean["AveBedrms"] / df_clean["AveRooms"]

    df_clean["MedInc"] = np.log1p(df_clean["MedInc"])
    df_clean["Population"] = np.log1p(df_clean["Population"])
    df_clean["AveBedrms"] = np.log1p(df_clean["AveBedrms"])

    df_clean["Dist_to_SF"] = np.sqrt(
        (df_clean["Latitude"] - SF_COORDS[0]) ** 2
        + (df_clean["Longitude"] - SF_COORDS[1]) ** 2
    )
    df_clean["Dist_to_LA"] = np.sqrt(
        (df_clean["Latitude"] - LA_COORDS[0]) ** 2
        + (df_clean["Longitude"] - LA_COORDS[1]) ** 2
    )
    df_clean["Dist_to_SJ"] = np.sqrt(
        (df_clean["Latitude"] - SJ_COORDS[0]) ** 2
        + (df_clean["Longitude"] - SJ_COORDS[1]) ** 2
    )

    features = df_clean.drop("MedHouseVal", axis=1)
    target = df_clean["MedHouseVal"].copy()

    X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
        features, target, test_size=0.2, random_state=SEED
    )

    scaler = sklearn.preprocessing.StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return df_clean, X_train_scaled, X_test_scaled, y_train, y_test


df_clean, X_train_scaled, X_test_scaled, y_train, y_test = prepare_data()
print(f"Datensatzgröße: {df_clean.shape}")

Datensatzgröße: (18570, 13)


# 3. Architektur-Experimente
Die folgenden Modelle werden mit denselben Trainings- und Evaluierungsparametern verglichen, um die Auswirkung unterschiedlicher Schichtgrößen zu untersuchen.

In [ ]:

def build_model(architecture, dropout_rate=0.2, l2_penalty=0.001):
    model = keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    ])

    for units in architecture:
        model.add(
            keras.layers.Dense(
                units,
                activation="relu",
                kernel_regularizer=keras.regularizers.l2(l2_penalty),
            )
        )
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Dropout(dropout_rate))

    model.add(keras.layers.Dense(1, activation="linear"))
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


def train_and_evaluate(architecture, epochs=300, batch_size=64):
    model = build_model(architecture)
    model.fit(
        X_train_scaled,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
        ],
        verbose=0,
    )

    test_scores = model.evaluate(X_test_scaled, y_test, verbose=0)
    return {
        "architecture": architecture,
        "mse": test_scores[0],
        "mae": test_scores[1],
    }


results = train_and_evaluate([128, 64, 32])
print(f"Architektur: {results['architecture']}")
print(f"MSE: {results['mse']:.2f}")
print(f"MAE: {results['mae']:.2f}")

Epoch 1/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3.0564 - mae: 1.4155 - val_loss: 1.1205 - val_mae: 0.7617 - learning_rate: 0.0010
Epoch 2/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0739 - mae: 0.7565 - val_loss: 0.4428 - val_mae: 0.4075 - learning_rate: 0.0010
Epoch 3/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7620 - mae: 0.6192 - val_loss: 0.3748 - val_mae: 0.3699 - learning_rate: 0.0010
Epoch 4/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6622 - mae: 0.5703 - val_loss: 0.3620 - val_mae: 0.3592 - learning_rate: 0.0010
Epoch 5/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5887 - mae: 0.5296 - val_loss: 0.3540 - val_mae: 0.3538 - learning_rate: 0.0010
Epoch 6/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5374 - mae: 0.4981 - val_loss: 0.3464 - val_mae: 0.3516 - learning_rate: 0.0010
Epoch 7/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5043 - mae: 0.4770 - val_loss: 0.3400 - val_mae: 0.3507 - learning_rate: 0.0010

In [ ]:
results = train_and_evaluate([128, 64, 32, 16])
print(f"Architektur: {results['architecture']}")
print(f"MSE: {results['mse']:.2f}")
print(f"MAE: {results['mae']:.2f}")

Epoch 1/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3.4569 - mae: 1.5090 - val_loss: 1.8933 - val_mae: 1.1384 - learning_rate: 0.0010
Epoch 2/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3436 - mae: 0.8520 - val_loss: 0.5998 - val_mae: 0.5043 - learning_rate: 0.0010
Epoch 3/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.9084 - mae: 0.6653 - val_loss: 0.4400 - val_mae: 0.4034 - learning_rate: 0.0010
Epoch 4/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7334 - mae: 0.5881 - val_loss: 0.4000 - val_mae: 0.3837 - learning_rate: 0.0010
Epoch 5/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6611 - mae: 0.5492 - val_loss: 0.3781 - val_mae: 0.3685 - learning_rate: 0.0010
Epoch 6/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6139 - mae: 0.5288 - val_loss: 0.3693 - val_mae: 0.3655 - learning_rate: 0.0010
Epoch 7/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5523 - mae: 0.4939 - val_loss: 0.3653 - val_mae: 0.3621 - learning_rate: 0.0010

In [ ]:
results = train_and_evaluate([128, 64, 32, 8])
print(f"Architektur: {results['architecture']}")
print(f"MSE: {results['mse']:.2f}")
print(f"MAE: {results['mae']:.2f}")

Epoch 1/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3.7850 - mae: 1.6075 - val_loss: 1.9889 - val_mae: 1.1400 - learning_rate: 0.0010
Epoch 2/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.7268 - mae: 1.0077 - val_loss: 0.7329 - val_mae: 0.5855 - learning_rate: 0.0010
Epoch 3/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.9590 - mae: 0.6959 - val_loss: 0.4377 - val_mae: 0.4066 - learning_rate: 0.0010
Epoch 4/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7853 - mae: 0.6235 - val_loss: 0.3907 - val_mae: 0.3809 - learning_rate: 0.0010
Epoch 5/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6792 - mae: 0.5740 - val_loss: 0.3717 - val_mae: 0.3680 - learning_rate: 0.0010
Epoch 6/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6187 - mae: 0.5445 - val_loss: 0.3632 - val_mae: 0.3634 - learning_rate: 0.0010
Epoch 7/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5737 - mae: 0.5170 - val_loss: 0.3591 - val_mae: 0.3602 - learning_rate: 0.0010

In [ ]:
results = train_and_evaluate([32, 16, 8])
print(f"Architektur: {results['architecture']}")
print(f"MSE: {results['mse']:.2f}")
print(f"MAE: {results['mae']:.2f}")

Epoch 1/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.8246 - mae: 1.6638 - val_loss: 2.0281 - val_mae: 1.2509 - learning_rate: 0.0010
Epoch 2/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.7479 - mae: 1.0359 - val_loss: 0.7278 - val_mae: 0.6335 - learning_rate: 0.0010
Epoch 3/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.9664 - mae: 0.7201 - val_loss: 0.3945 - val_mae: 0.4273 - learning_rate: 0.0010
Epoch 4/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7648 - mae: 0.6362 - val_loss: 0.3705 - val_mae: 0.4139 - learning_rate: 0.0010
Epoch 5/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6627 - mae: 0.5914 - val_loss: 0.3492 - val_mae: 0.3991 - learning_rate: 0.0010
Epoch 6/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5869 - mae: 0.5540 - val_loss: 0.3509 - val_mae: 0.4008 - learning_rate: 0.0010
Epoch 7/300
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5802 - mae: 0.5498 - val_loss: 0.3366 - val_mae: 0.3927 - learning_rate: 0.0010

In [ ]:
results = train_and_evaluate([1024, 512], batch_size=128)
print(f"Architektur: {results['architecture']}")
print(f"MSE: {results['mse']:.2f}")
print(f"MAE: {results['mae']:.2f}")

Epoch 1/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 3.6033 - mae: 1.2560 - val_loss: 1.4265 - val_mae: 0.6837 - learning_rate: 0.0010
Epoch 2/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6028 - mae: 0.7254 - val_loss: 1.3259 - val_mae: 0.6552 - learning_rate: 0.0010
Epoch 3/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4041 - mae: 0.6490 - val_loss: 1.1859 - val_mae: 0.5812 - learning_rate: 0.0010
Epoch 4/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.2293 - mae: 0.5806 - val_loss: 1.0655 - val_mae: 0.5079 - learning_rate: 0.0010
Epoch 5/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.1340 - mae: 0.5451 - val_loss: 0.9361 - val_mae: 0.4335 - learning_rate: 0.0010
Epoch 6/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.0449 - mae: 0.5103 - val_loss: 0.8443 - val_mae: 0.3920 - learning_rate: 0.0010
Epoch 7/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.9818 - mae: 0.4893 - val_loss: 0.7885 - val_mae: 0.3583 - learning_rate: 0.0010
Epoch 8/300
9

In [ ]:
# Die Modellvergleichszellen können nacheinander ausgeführt werden, um die Ergebnisse direkt zu vergleichen.